# Week 4 — BERTopic (Student B: Facebook + White House)

Run in Colab. **You only edit the `EDIT ME` cell** (input/output folders + which corpora). Reads the input `*_vader_roberta.csv` from `IN_DIR`, writes `*_topics.csv` + saved models to `OUT_DIR`.

Topic written back as `topic_bertopic` (+ `topic_label_auto`); short rows (<5 words) keep the sentinel `excluded_short`, never deleted.

| key | corpus | ~rows |
|---|---|---|
| `facebook_posts` | FB posts | 952 |
| `facebook_comments` | FB comments | 59,736 |
| `whitehouse` | WH posts(12)+comments(2,193), combined | 2,205 |
| `reddit_posts` | Reddit posts | 3,418 |
| `reddit_comments` | Reddit comments | 122,026 |

**Runtime -> Change runtime type -> T4 GPU** before running.

In [ ]:
# 1. Install (pulls umap-learn, hdbscan, sentence-transformers)
!pip -q install bertopic

In [ ]:
# ===================================================================
#  EDIT ME  --  the paths you normally change
# ===================================================================
# Folder that HOLDS the input *_vader_roberta.csv files  (note: capital "Roberta")
IN_DIR  = '/content/drive/MyDrive/SentimentofHurricanes/Summer2026/Week3 Deliverables/data/Roberta'

# Where to WRITE topic outputs + saved models (new Week 4 folder -- create it in Drive first)
OUT_DIR = '/content/drive/MyDrive/SentimentofHurricanes/Summer2026/Week4 Deliverables'

# Which corpora to model. Valid keys:
#   facebook_posts, facebook_comments, whitehouse, reddit_posts, reddit_comments
RUN = ["facebook_posts", "facebook_comments", "whitehouse"]
# ===================================================================
#  NOTE: these files are under "Shared with me", which Colab CANNOT open by path
#  until you add a shortcut: in Drive, right-click "SentimentofHurricanes" ->
#  Organize -> Add shortcut to Drive -> My Drive. Then the MyDrive paths resolve.
# ===================================================================

In [ ]:
# 2. Mount Drive + set up the output folders
from google.colab import drive
drive.mount('/content/drive')

import os
MODELS = f"{OUT_DIR}/models/bertopic"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(MODELS, exist_ok=True)

# should list the input files
!ls "{IN_DIR}" | grep -E '_vader_roberta\\.csv$'

In [ ]:
# 3. Imports + fixed knobs
import numpy as np, pandas as pd
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP

EMBED_MODEL = "all-MiniLM-L6-v2"
MIN_WORDS   = 5
TEXT_COL    = "text"
RANDOM_SEED = 42

def make_umap():
    return UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=RANDOM_SEED)

def make_vectorizer():
    return CountVectorizer(stop_words="english", ngram_range=(1, 2))

print("[config] embed =", EMBED_MODEL, "| min_words =", MIN_WORDS)

In [ ]:
# 4. Helpers
def add_wordcount(df):
    df = df.copy()
    df['n_words'] = df[TEXT_COL].fillna("").astype(str).str.split().str.len()
    return df

def run_bertopic(df, name, min_topic_size):
    """Fit BERTopic on the >=MIN_WORDS subset; keep all rows, short ones get 'excluded_short'."""
    df = add_wordcount(df)
    sub = df[df['n_words'] >= MIN_WORDS]
    docs = sub[TEXT_COL].fillna("").astype(str).tolist()
    print(f"\n=== {name} ===")
    print(f"[rows] total {len(df)} | modeled {len(sub)} | dropped <{MIN_WORDS} words {len(df)-len(sub)}")
    print(f"[param] min_topic_size = {min_topic_size}")
    model = BERTopic(embedding_model=EMBED_MODEL, umap_model=make_umap(),
                     vectorizer_model=make_vectorizer(), min_topic_size=min_topic_size,
                     calculate_probabilities=False, verbose=True)
    topics, _ = model.fit_transform(docs)
    info = model.get_topic_info()
    print(f"[result] {(info['Topic']>=0).sum()} topics | {100*(np.array(topics)==-1).mean():.1f}% in -1 outliers")
    print(info.head(15).to_string(index=False))
    df['topic_bertopic'] = 'excluded_short'
    df.loc[sub.index, 'topic_bertopic'] = topics
    name_map = dict(zip(info['Topic'], info['Name']))
    df['topic_label_auto'] = df['topic_bertopic'].map(
        lambda t: name_map.get(t, '') if t != 'excluded_short' else 'excluded_short')
    return model, df

In [ ]:
# 5. Job table (advanced -- tune min_topic_size here if topics look bad)
JOBS = {
    "facebook_posts":    dict(mode="single",   file="facebook_posts_vader_roberta.csv",            min_topic_size=10),
    "facebook_comments": dict(mode="single",   file="facebook_comments_vader_roberta.csv",         min_topic_size=150),
    "reddit_posts":      dict(mode="single",   file="reddit_relevant_posts_vader_roberta.csv",      min_topic_size=30),
    "reddit_comments":   dict(mode="single",   file="reddit_relevant_comments_vader_roberta.csv",   min_topic_size=250),
    "whitehouse":        dict(mode="combined", files=["whitehouse_threads_posts_vader_roberta.csv",
                                                       "whitehouse_threads_comments_vader_roberta.csv"], min_topic_size=20),
}
print("[run]", RUN)

In [ ]:
# 6. Quickstart sanity check -- 500 rows from the heaviest file in RUN
qs_file = next((JOBS[k]['file'] for k in RUN if JOBS[k]['mode']=='single' and 'comments' in JOBS[k]['file']),
               next((JOBS[k]['file'] for k in RUN if JOBS[k]['mode']=='single'), None))
if qs_file:
    peek = add_wordcount(pd.read_csv(f"{IN_DIR}/{qs_file}"))
    peek = peek[peek['n_words'] >= MIN_WORDS]
    peek = peek.sample(min(500, len(peek)), random_state=RANDOM_SEED)
    qs = BERTopic(embedding_model=EMBED_MODEL, umap_model=make_umap(),
                  vectorizer_model=make_vectorizer(), min_topic_size=10, verbose=False)
    qs.fit_transform(peek[TEXT_COL].fillna("").astype(str).tolist())
    print(f"[quickstart on {qs_file}]")
    print(qs.get_topic_info().head(10).to_string(index=False))

## Tune `min_topic_size` (in the JOBS table)
Reddit comments (~122k) ~250 | FB comments (~60k) ~150 | Reddit posts (~3.4k) ~30 | WH (~2.2k) ~20 | FB posts (~950) ~10.

If the **-1 outlier** bucket swallows most rows, lower the value. Sweep `{15,20,30,50,...}` and keep what gives interpretable, non-redundant topics. Record the value you used.

In [ ]:
# 7. Run every job in RUN
models, out_csvs = {}, {}
for key in RUN:
    job = JOBS[key]
    if job['mode'] == 'single':
        df = pd.read_csv(f"{IN_DIR}/{job['file']}")
        m, out = run_bertopic(df, key, job['min_topic_size'])
        models[key] = m
        out_csvs[job['file'].replace('.csv', '_topics.csv')] = out
    else:  # combined: model together, split back per source file
        parts = []
        for fn in job['files']:
            d = pd.read_csv(f"{IN_DIR}/{fn}"); d['__src_file'] = fn
            parts.append(d)
        combo = pd.concat(parts, ignore_index=True)
        m, combo_out = run_bertopic(combo, key, job['min_topic_size'])
        models[key] = m
        for fn in job['files']:
            piece = combo_out[combo_out['__src_file'] == fn].drop(columns='__src_file')
            out_csvs[fn.replace('.csv', '_topics.csv')] = piece

In [ ]:
# 8. Write topic CSVs to OUT_DIR (new *_topics.csv -- never overwrite the scored inputs)
for fname, df in out_csvs.items():
    df.drop(columns=['n_words'], errors='ignore').to_csv(f"{OUT_DIR}/{fname}", index=False)
    print(f"[saved] {fname}  ({len(df)} rows)")

In [ ]:
# 9. Save models for reproducibility (Week 8)
for key, m in models.items():
    m.save(f"{MODELS}/{key}", serialization="safetensors", save_ctfidf=True)
    print(f"[saved model] {key}")

In [ ]:
# 10. QC -- row preservation + sentiment smell test on the largest output
for fname, df in out_csvs.items():
    short = (df['topic_bertopic'] == 'excluded_short').sum()
    print(f"{fname:58s} rows {len(df):>7} | excluded_short {short}")
biggest = max(out_csvs, key=lambda k: len(out_csvs[k]))
d = out_csvs[biggest]; d = d[d['topic_bertopic'] != 'excluded_short']
top = d['topic_bertopic'].value_counts().head(6).index
print(f"\n[smell test] {biggest} -- topic x vader_label:")
print(pd.crosstab(d[d['topic_bertopic'].isin(top)]['topic_bertopic'],
                  d[d['topic_bertopic'].isin(top)]['vader_label']))

In [ ]:
# 11. Run-log summary -- paste into docs/week4/bertopic_run_log.md (add chosen min_topic_size + draft labels)
import bertopic, sentence_transformers
print("bertopic", bertopic.__version__, "| sentence-transformers", sentence_transformers.__version__)
for key, m in models.items():
    info = m.get_topic_info()
    print(f"\n--- {key}: {(info['Topic']>=0).sum()} topics ---")
    print(info[['Topic','Count','Name']].head(12).to_string(index=False))

## Next steps (do NOT do alone)
- **Draft** topic labels from each model's top words; reconcile jointly into the shared codebook before any cross-source comparison.
- **Fallback** for mushy topics: `seed_topic_list=[["evacuation","evacuate","shelter"],["fema","aid","relief"],["power","outage","grid"],["forecast","track","landfall"]]`, or sklearn LDA.
- **Joint / later:** topic x source_type and x hurricane, chi-square, figures -- after all five models + the reconciled codebook are ready.